# 투구 제구 확률 실험 워크벤치

Public 챔피언 **v17 = 1093.3213473808**을 기준으로 데이터 점검, 로컬 평가, pressure EB screen, 최종 학습, 재패키징과 검증을 순서대로 호출합니다. 계산 로직은 `src/` 모듈이 단일 원본이며 이 노트북은 코드를 복제하지 않는 실행 계층입니다.

처음에는 `DRY_RUN=True`로 전체 셀을 실행해 명령과 입력 경로만 확인하세요. 자세한 절차는 `docs/EXPERIMENT_WORKFLOW.md`를 따릅니다.

In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src').is_dir() and (candidate / 'research').is_dir():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
print(f'project: {PROJECT_ROOT}')
print(f'python : {sys.executable}')
print(f'version: {sys.version.split()[0]}')

## 실행 스위치

- `DRY_RUN`: `True`이면 명령만 출력합니다.
- `RUN_HEAVY`: screen·학습·강건 평가를 허용합니다.
- `RUN_PACKAGING`: 새 ZIP 생성과 end-to-end 검증을 허용합니다.

무거운 셀은 필요한 private artifact가 모두 준비된 뒤에만 켜세요. `RUN_ID`는 팀 실험 기록에서 고유해야 합니다.

In [ ]:
DRY_RUN = True
RUN_TESTS = True
RUN_HEAVY = False
RUN_PACKAGING = False
RUN_ID = 'v19_team_20260816_01'

V17_PUBLIC = 1093.3213473808
TARGET_PUBLIC = 1100.0
V17_SELECTED = 'multi_pitcher_batter_hand_pressure_d1_a3200_w1.5'
V17_SCREEN_DIR = 'v16_multiseason_20260815_02'
V14_SHA256 = 'AC8E135FBA8DBF41E331A8F4E2AAA658F0AF6675DD5F3B6C89EF51E9BC02977E'

print(f'gap to target: {TARGET_PUBLIC - V17_PUBLIC:.10f}')

In [ ]:
def command_text(command):
    return subprocess.list2cmdline([str(part) for part in command])


def run_command(command, *, enabled, label):
    command = [str(part) for part in command]
    print(f'[{label}] {command_text(command)}')
    if DRY_RUN:
        print('  DRY_RUN: 실행하지 않음')
        return None
    if not enabled:
        print('  실행 스위치가 꺼져 있음')
        return None
    return subprocess.run(command, cwd=PROJECT_ROOT, check=True)


def run_module(module, *args, enabled, label):
    return run_command(
        [sys.executable, '-m', module, *args], enabled=enabled, label=label
    )

## 입력 준비 상태

존재 여부만 확인하며 데이터 내용이나 모델을 출력하지 않습니다. `submit*.zip`과 `artifacts/`는 Git 대상이 아닙니다.

In [ ]:
required_paths = {
    'train data': Path('data/train.csv'),
    'test data': Path('data/test.csv'),
    'v14 parent': Path('submit_v14.zip'),
    'v17 champion': Path('submit_v17.zip'),
    'v17 screen cache': Path('artifacts') / V17_SCREEN_DIR,
    'v17 residual artifact': Path('artifacts/v17_residual_final_20260815'),
}
for name, path in required_paths.items():
    state = 'READY' if path.exists() else 'MISSING'
    print(f'{state:7}  {name:24}  {path}')

## 제출 이력 확인

`reports/submissions.csv`의 마지막 기록을 읽어 현재 문서와 제출 이력이 일치하는지 확인합니다.

In [ ]:
with Path('research/reports/submissions.csv').open(encoding='utf-8-sig', newline='') as stream:
    submission_rows = list(csv.DictReader(stream))

for row in submission_rows[-5:]:
    print({key: row.get(key) for key in ('submission_id', 'name', 'public_score', 'submitted_at')})

## 데이터 비의존 검증

저장소 감사와 핵심 통계·추론·노트북 테스트를 실행합니다. `DRY_RUN=False`일 때 실제로 동작합니다.

In [ ]:
run_command(
    [sys.executable, 'scripts/audit_repository.py', '--include-untracked'],
    enabled=RUN_TESTS,
    label='repository audit',
)
run_module(
    'pytest', '-q',
    'tests/test_metrics.py',
    'tests/test_local_scorecard.py',
    'tests/test_robust_local_evaluation.py',
    'tests/test_v16_residual.py',
    'tests/test_experiment_notebook.py',
    enabled=RUN_TESTS, label='data-free tests',
)

## 후보 축 screen

출력은 고유한 `artifacts/` 하위 폴더에 기록합니다. 최고 한 점이 아니라 최근 전이 최소 gain과 의존성 강건성을 함께 판정합니다.

In [ ]:
run_module(
    'src.archive.v16_residual_calibration_screen',
    '--output-dir', f'artifacts/{RUN_ID}_residual_calibration',
    enabled=RUN_HEAVY, label='residual calibration screen',
)
run_module(
    'src.archive.v16_multiseason_screen',
    '--output-dir', f'artifacts/{RUN_ID}_multiseason',
    enabled=RUN_HEAVY, label='multi-season EB screen',
)

## v17 선택 recipe 강건 평가

기존 확정 보고서를 덮어쓰지 않고 `RUN_ID`가 붙은 새 보고서를 만듭니다. `--artifact-dir`는 `artifacts/` 아래 디렉터리 이름입니다.

In [ ]:
run_module(
    'src.archive.evaluate_v16_robust',
    '--selected', V17_SELECTED,
    '--artifact-dir', V17_SCREEN_DIR,
    '--parent-f-alpha', '0.15',
    '--output-json', f'research/reports/{RUN_ID}_robust.json',
    '--output-markdown', f'research/reports/{RUN_ID}_robust.md',
    enabled=RUN_HEAVY, label='dependence-aware robust evaluation',
)

## 최종 pressure residual 학습

v17 recipe를 재현하는 예입니다. 새 연구에서는 값을 바꾸기 전에 screen과 승격 기준을 통과해야 합니다.

In [ ]:
FINAL_ARTIFACT_DIR = f'{RUN_ID}_residual_final'
run_module(
    'src.archive.train_v16_residual',
    '--output-dir', f'artifacts/{FINAL_ARTIFACT_DIR}',
    '--group', 'pitcher_batter_hand_pressure',
    '--decay', '1',
    '--alpha', '3200',
    '--correction-weight', '1.5',
    '--candidate-name', RUN_ID,
    enabled=RUN_HEAVY, label='fit final pressure EB spec',
)

## v17 구조 재패키징과 end-to-end 검증

현재 패키저는 v14 부모에 residual spec 한 개를 추가합니다. v17 위에 다른 overlay를 중첩하는 도구가 아닙니다. 아래 출력명은 기존 챔피언을 덮어쓰지 않습니다.

In [ ]:
REBUILD_ZIP = f'submit_{RUN_ID}_rebuild.zip'
run_module(
    'src.archive.package_v16_residual',
    '--parent', 'submit_v14.zip',
    '--output', REBUILD_ZIP,
    '--artifact-dir', FINAL_ARTIFACT_DIR,
    '--expected-parent-sha256', V14_SHA256,
    enabled=RUN_PACKAGING, label='package candidate',
)
run_module(
    'src.archive.validate_v16_residual',
    '--candidate', REBUILD_ZIP,
    '--parent', 'submit_v14.zip',
    '--report', f'research/reports/{RUN_ID}_validation.md',
    enabled=RUN_PACKAGING, label='validate candidate package',
)

## 실험 종료 체크리스트

1. 후보 가설과 야구적 근거를 보고서에 기록합니다.
2. 시즌·도메인·월별 gain과 모든 bootstrap/Reality Check를 검토합니다.
3. 부모와 후보 ZIP의 SHA-256, runtime, memory, batch invariance를 기록합니다.
4. 코드·설정·요약 보고서만 Git에 올리고 데이터·OOF·모델·ZIP은 제외합니다.
5. 팀 리뷰와 승격 승인을 받은 후보만 지정 담당자가 제출합니다.
6. 결과를 `reports/submissions.csv`와 `docs/PROJECT_STATUS.md`에 반영합니다.